In [ ]:
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split


all_csv_paths = glob.glob("data/raw/*.csv")


file_labels = []
for fpath in all_csv_paths:

    df_flag = pd.read_csv(fpath, usecols=["Seizure [bool]"])

    has_seizure = df_flag["Seizure [bool]"].any()
    file_labels.append(int(has_seizure))

# 3) Stratified split: 80% train, 20% test
#    We pass `stratify=file_labels` so that both subsets keep the same proportion of 0’s and 1’s
train_paths, test_paths = train_test_split(
    all_csv_paths,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=file_labels
)

# 4) Ensure the output directory exists
os.makedirs("data/splits", exist_ok=True)

# 5) Write train list to data/splits/train_files.txt
train_txt = os.path.join("data/splits", "train_files.txt")
with open(train_txt, "w") as f_train:
    for path in train_paths:
        f_train.write(f"{path}\n")

# 6) Write test list to data/splits/test_files.txt
test_txt = os.path.join("data/splits", "test_files.txt")
with open(test_txt, "w") as f_test:
    for path in test_paths:
        f_test.write(f"{path}\n")

# 7) Print a quick summary
n_total = len(all_csv_paths)
n_train = len(train_paths)
n_test = len(test_paths)
print(f"Total files: {n_total}")
print(
    f"  → Train: {n_train} (contains {sum([int(l == 1) for i, l in enumerate(file_labels) if all_csv_paths[i] in set(train_paths)])} seizure‐files)")
print(
    f"  → Test : {n_test} (contains {sum([int(l == 1) for i, l in enumerate(file_labels) if all_csv_paths[i] in set(test_paths)])} seizure‐files)")
print(f"Split lists saved as:\n  • {train_txt}\n  • {test_txt}")

In [ ]:

# def conv_bn(in_channels: int, out_channels: int, kernel_size: int) -> nn.Sequential:
#     padding = (kernel_size - 1) // 2
#     return nn.Sequential(
#         nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding, bias=False),
#         nn.BatchNorm1d(out_channels),
#         nn.ReLU(inplace=True),
#     )


# class SqueezeExcite1D(nn.Module):
#     def __init__(self, channels: int, reduction: int = 16):
#         super().__init__()
#         self.avg_pool = nn.AdaptiveAvgPool1d(1)
#         self.fc = nn.Sequential(
#             nn.Conv1d(channels, channels // reduction, kernel_size=1),
#             nn.ReLU(inplace=True),
#             nn.Conv1d(channels // reduction, channels, kernel_size=1),
#             nn.Sigmoid(),
#         )

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         weights = self.avg_pool(x)
#         weights = self.fc(weights)
#         return x * weights


# class InceptionBlock(nn.Module):
#     def __init__(
#         self,
#         in_channels: int,
#         out_channels: int,
#         kernel_sizes: List[int],
#         bottleneck_channels: int,
#         use_se: bool,
#     ):
#         super().__init__()
#         self.use_se = use_se

#         if bottleneck_channels > 0:
#             self.bottleneck = nn.Conv1d(in_channels, bottleneck_channels, kernel_size=1, bias=False)
#         else:
#             self.bottleneck = None
#             bottleneck_channels = in_channels

#         self.branches = nn.ModuleList(
#             [conv_bn(bottleneck_channels, out_channels, k) for k in kernel_sizes]
#         )

#         self.pool_branch = nn.Sequential(
#             nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
#             conv_bn(in_channels, out_channels, kernel_size=1),
#         )

#         total_channels = (len(kernel_sizes) + 1) * out_channels
#         self.bn = nn.BatchNorm1d(total_channels)
#         self.relu = nn.ReLU(inplace=True)

#         if use_se:
#             self.se = SqueezeExcite1D(total_channels)

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         y = x
#         if self.bottleneck is not None:
#             y = self.bottleneck(x)

#         branch_outputs = [branch(y) for branch in self.branches]
#         branch_outputs.append(self.pool_branch(x))

#         min_length = min(out.size(-1) for out in branch_outputs)
#         trimmed = [out[:, :, :min_length] for out in branch_outputs]

#         out = torch.cat(trimmed, dim=1)
#         out = self.bn(out)

#         if self.use_se:
#             out = self.se(out)

#         return self.relu(out)


# class InceptionTimeSE(nn.Module):
#     def __init__(
#         self,
#         n_blocks: int = 6,
#         in_channels: int = 1,
#         n_classes: int = 1,
#         out_channels: int = 32,
#         bottleneck_channels: int = 32,
#         kernel_sizes: Optional[List[int]] = None,
#         use_se: bool = True,
#     ):
#         super().__init__()
#         if kernel_sizes is None:
#             kernel_sizes = [10, 20, 40]

#         layers = []
#         current_channels = in_channels

#         for _ in range(n_blocks):
#             block = InceptionBlock(
#                 in_channels=current_channels,
#                 out_channels=out_channels,
#                 kernel_sizes=kernel_sizes,
#                 bottleneck_channels=bottleneck_channels,
#                 use_se=use_se,
#             )
#             layers.append(block)
#             current_channels = (len(kernel_sizes) + 1) * out_channels

#         self.encoder = nn.Sequential(*layers)
#         self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
#         self.classifier = nn.Linear(current_channels, n_classes)

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         x = self.encoder(x)
#         x = self.global_avg_pool(x).squeeze(-1)
#         return self.classifier(x).squeeze(-1)